In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

print("Starting full system reconstruction with robust label parsing...")

# =====================================================================
# 1. DEFINE PATHS
# =====================================================================
base_path = '/kaggle/input/datasets/abtinzandi/obstacle-detection-dataset/ROD-Dataset/dataset'
train_images_dir = os.path.join(base_path, 'train/images')
train_labels_dir = os.path.join(base_path, 'train/labels')
val_images_dir = os.path.join(base_path, 'valid/images')
val_labels_dir = os.path.join(base_path, 'valid/labels')

# =====================================================================
# 2. BUILD THE ARCHITECTURE
# =====================================================================
def build_detector():
    base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base_model.trainable = False  
    
    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    
    box_output = layers.Dense(4, activation='sigmoid', name='bounding_box')(x)
    class_output = layers.Dense(2, activation='softmax', name='class_label')(x)
    
    return models.Model(inputs=inputs, outputs=[box_output, class_output])

model = build_detector()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={'bounding_box': 'mean_squared_error', 'class_label': 'categorical_crossentropy'},
    metrics={'class_label': 'accuracy'}
)

# =====================================================================
# 3. CONSTRUCT FAULT-TOLERANT DATA PIPELINE
# =====================================================================
def get_filepaths(images_dir, labels_dir):
    img_files = sorted([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    img_paths = [os.path.join(images_dir, f) for f in img_files]
    lbl_paths = [os.path.join(labels_dir, os.path.splitext(f)[0] + '.txt') for f in img_files]
    return img_paths, lbl_paths

train_img_paths, train_lbl_paths = get_filepaths(train_images_dir, train_labels_dir)
val_img_paths, val_lbl_paths = get_filepaths(val_images_dir, val_labels_dir)

def parse_element_safe(img_path, lbl_path):
    # Process image cleanly
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img = img / 255.0
    
    # Read label file content safely
    label_content = tf.io.read_file(lbl_path)
    lines = tf.strings.split(label_content, '\n')
    
    # Check if the label file is completely empty or missing content
    num_lines = tf.shape(lines)[0]
    
    def process_valid_label():
        first_line = tf.strings.split(lines[0], ' ')
        
        # Guard condition: if line split doesn't give us 5 elements, use default fallback
        return tf.cond(
            tf.greater_equal(tf.shape(first_line)[0], 5),
            lambda: (
                tf.strings.to_number(first_line[0], out_type=tf.int32),
                tf.strings.to_number(first_line[1], out_type=tf.float32),
                tf.strings.to_number(first_line[2], out_type=tf.float32),
                tf.strings.to_number(first_line[3], out_type=tf.float32),
                tf.strings.to_number(first_line[4], out_type=tf.float32)
            ),
            lambda: (0, 0.0, 0.0, 0.0, 0.0)
        )
        
    class_id, x_center, y_center, w, h = tf.cond(
        tf.greater(num_lines, 0),
        process_valid_label,
        lambda: (0, 0.0, 0.0, 0.0, 0.0)
    )
    
    # Map coordinates safely
    xmin = x_center - (w / 2.0)
    ymin = y_center - (h / 2.0)
    xmax = x_center + (w / 2.0)
    ymax = y_center + (h / 2.0)
    
    box_target = tf.stack([xmin, ymin, xmax, ymax])
    class_target = tf.cond(tf.equal(class_id, 0), lambda: tf.constant([1.0, 0.0]), lambda: tf.constant([0.0, 1.0]))
    
    return img, {'bounding_box': box_target, 'class_label': class_target}

# Convert elements into active batched datasets
train_dataset = tf.data.Dataset.from_tensor_slices((train_img_paths, train_lbl_paths))
train_dataset = train_dataset.map(parse_element_safe, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=100).batch(32).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((val_img_paths, val_lbl_paths))
val_dataset = val_dataset.map(parse_element_safe, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

print(f"\nReconstruction complete! Streaming pipeline running smoothly over {len(train_img_paths)} items...")

# =====================================================================
# 4. EXECUTE TRAINING LOOP
# =====================================================================
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    verbose=1
)

model.save('/kaggle/working/tf_obstacle_detector.h5')
print("\n[SUCCESS] Model successfully trained and saved to /kaggle/working/tf_obstacle_detector.h5!")

In [ ]:
import cv2
import numpy as np
import tensorflow as tf

# 1. Load your newly trained model weights
trained_model = tf.keras.models.load_model('/kaggle/working/tf_obstacle_detector.h5')

# 2. Use your exact copied video path
video_path = '/kaggle/input/datasets/abuzarjamil/test-video/WhatsApp Video 2026-06-22 at 11.05.49 PM.mp4'
cap = cv2.VideoCapture(video_path)

# Extract properties of your video
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS)) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30

# 3. Setup the video writer to save the result in Kaggle's working directory
output_path = '/kaggle/working/output_detected.mp4'
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

class_names = ['Person', 'Cone']
print("Processing video frames and applying tracking boxes...")

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    # Convert and scale the frame to match what the network expects
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (224, 224))
    img_batch = np.expand_dims(img_resized.astype(np.float32) / 255.0, axis=0)
    
    # Run predictions through both heads simultaneously
    box_preds, class_preds = trained_model.predict(img_batch, verbose=0)
    class_idx = np.argmax(class_preds[0])
    confidence = class_preds[0][class_idx]
    
    # Only draw if the model is confident
    if confidence > 0.4:
        # Denormalize coordinates back to the original video frame size
        xmin, ymin, xmax, ymax = box_preds[0]
        start_point = (int(xmin * frame_width), int(ymin * frame_height))
        end_point = (int(xmax * frame_width), int(ymax * frame_height))
        
        # Color coding: Green for Person, Orange for Cone
        color = (0, 255, 0) if class_idx == 0 else (0, 165, 255)
        
        # Overlay box and label graphics onto the original frame
        cv2.rectangle(frame, start_point, end_point, color, 3)
        cv2.putText(frame, f"{class_names[class_idx]} {confidence:.2f}", 
                    (start_point[0], max(start_point[1] - 10, 20)), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        
    out.write(frame)
    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Processed {frame_count} frames...")

cap.release()
out.release()
print("\n[SUCCESS] Video processing finished completely!")
print("Your tracked video is saved as: /kaggle/working/output_detected.mp4")

In [ ]:
import os
print(os.listdir('/kaggle/input/datasets/abuzarjamil/test-video'))

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models

# =====================================================================
# 1. BUILD THE REFINED, MULTI-SCALE LOCALIZATION ARCHITECTURE
# =====================================================================
def build_detector():
    base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base_model.trainable = False  # Keep shared features locked
    
    inputs = layers.Input(shape=(224, 224, 3))
    x_shared = base_model(inputs, training=False)
    x_pool = layers.GlobalAveragePooling2D()(x_shared)
    x_shared_dense = layers.Dense(128, activation='relu')(x_pool)
    x_shared_dense = layers.Dropout(0.2)(x_shared_dense)
    
    # --- Localization Branch Refinement ---
    # NEW: We are adding specific convolutional layers *after* the base 
    # but *before* the dense layer. These layers can focus on smaller, 
    # localized features, such as the edge of the cone, which the 
    # shared features might blur.
    
    # Focus on the highest resolution shared feature map (this may need adjustments 
    # based on the specific layers of the base, but this serves as a good default example)
    feature_map_high_res = base_model.layers[-20].output # Approximate a key layer
    
    x_loc_refine = layers.Conv2D(32, (3,3), padding='same', activation='relu', name='loc_refine_conv1')(feature_map_high_res)
    x_loc_refine = layers.GlobalAveragePooling2D()(x_loc_refine) # Map localized features back
    
    # Combined refined and shared dense layers for bounding box prediction
    x_combined_box = layers.Concatenate()([x_shared_dense, x_loc_refine])
    x_combined_box = layers.Dense(64, activation='relu')(x_combined_box)

    # Head 1: Bounding Box (Uses the localized features)
    box_output = layers.Dense(4, activation='sigmoid', name='bounding_box')(x_combined_box)
    
    # Head 2: Classification (Can stick with the shared features)
    class_output = layers.Dense(2, activation='softmax', name='class_label')(x_shared_dense)
    
    return models.Model(inputs=inputs, outputs=[box_output, class_output])

# =====================================================================
# 2. IMPLEMENT DISTANCE-BASED IoU LOSS FOR LOCALIZATION ACCURACY
# =====================================================================
# This loss explicitly penalizes distance between the ground truth center
# and predicted center, which is excellent for tightening boxes.
def diou_loss(y_true, y_pred):
    true_boxes = y_true
    pred_boxes = y_pred
    
    t_xmin, t_ymin, t_xmax, t_ymax = tf.unstack(true_boxes, axis=-1)
    p_xmin, p_ymin, p_xmax, p_ymax = tf.unstack(pred_boxes, axis=-1)
    
    # Calculate IoU
    inter_xmin = tf.maximum(t_xmin, p_xmin)
    inter_ymin = tf.maximum(t_ymin, p_ymin)
    inter_xmax = tf.minimum(t_xmax, p_xmax)
    inter_ymax = tf.minimum(t_ymax, p_ymax)
    
    inter_area = tf.maximum(0.0, inter_xmax - inter_xmin) * tf.maximum(0.0, inter_ymax - inter_ymin)
    
    true_area = (t_xmax - t_xmin) * (t_ymax - t_ymin)
    pred_area = (p_xmax - p_xmin) * (p_ymax - p_ymin)
    union_area = true_area + pred_area - inter_area
    
    iou = inter_area / tf.maximum(union_area, 1e-6)
    
    # Calculate centers
    t_cx = (t_xmin + t_xmax) / 2.0
    t_cy = (t_ymin + t_ymax) / 2.0
    p_cx = (p_xmin + p_xmax) / 2.0
    p_cy = (p_ymin + p_ymax) / 2.0
    
    # Euclidean distance between centers (squared)
    center_dist_sq = tf.square(t_cx - p_cx) + tf.square(t_cy - p_cy)
    
    # Find the smallest enclosing box
    enclosing_xmin = tf.minimum(t_xmin, p_xmin)
    enclosing_ymin = tf.minimum(t_ymin, p_ymin)
    enclosing_xmax = tf.maximum(t_xmax, p_xmax)
    enclosing_ymax = tf.maximum(t_ymax, p_ymax)
    
    # Calculate the enclosing box's diagonal (squared)
    enc_diag_sq = tf.square(enclosing_xmax - enclosing_xmin) + tf.square(enclosing_ymax - enclosing_ymin)
    
    # DIoU penalty
    penalty = center_dist_sq / tf.maximum(enc_diag_sq, 1e-6)
    
    return 1.0 - iou + penalty

model = build_detector()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={'bounding_box': diou_loss, 'class_label': 'categorical_crossentropy'},
    metrics={'class_label': 'accuracy'}
)

print("\nModel reconstructed with refined localization pipeline and DIoU loss.")
# ... [Rest of the training, tf.data, and execution code identical to previous self-contained reconstruction cell]
# ... [Assuming all previous data streaming cells are available as provided earlier]

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

print("Starting system upgrade with refined localization and DIoU Loss...")

# =====================================================================
# 1. SET PATHS
# =====================================================================
base_path = '/kaggle/input/datasets/abtinzandi/obstacle-detection-dataset/ROD-Dataset/dataset'
train_images_dir = os.path.join(base_path, 'train/images')
train_labels_dir = os.path.join(base_path, 'train/labels')
val_images_dir = os.path.join(base_path, 'valid/images')
val_labels_dir = os.path.join(base_path, 'valid/labels')

# =====================================================================
# 2. BUILD THE UPGRADED DETECTOR ARCHITECTURE (Graph Safe)
# =====================================================================
def build_detector():
    base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base_model.trainable = False  
    
    inputs = layers.Input(shape=(224, 224, 3))
    x_shared = base_model(inputs, training=False) # Shape: (None, 7, 7, 1280)
    
    # Classification Branch
    x_class = layers.GlobalAveragePooling2D()(x_shared)
    x_class = layers.Dense(128, activation='relu')(x_class)
    x_class = layers.Dropout(0.2)(x_class)
    class_output = layers.Dense(2, activation='softmax', name='class_label')(x_class)
    
    # Upgraded Localization Branch (Spatial Refiner)
    # We add specialized Conv layers directly on top of the feature map to sharpen edges
    x_loc = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(x_shared)
    x_loc = layers.BatchNormalization()(x_loc)
    x_loc = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x_loc)
    x_loc = layers.GlobalAveragePooling2D()(x_loc)
    x_loc = layers.Dense(64, activation='relu')(x_loc)
    box_output = layers.Dense(4, activation='sigmoid', name='bounding_box')(x_loc)
    
    return models.Model(inputs=inputs, outputs=[box_output, class_output])

# =====================================================================
# 3. DISTANCE-BASED IoU LOSS FUNCTION
# =====================================================================
def diou_loss(y_true, y_pred):
    true_boxes = y_true
    pred_boxes = y_pred
    
    t_xmin, t_ymin, t_xmax, t_ymax = tf.unstack(true_boxes, axis=-1)
    p_xmin, p_ymin, p_xmax, p_ymax = tf.unstack(pred_boxes, axis=-1)
    
    inter_xmin = tf.maximum(t_xmin, p_xmin)
    inter_ymin = tf.maximum(t_ymin, p_ymin)
    inter_xmax = tf.minimum(t_xmax, p_xmax)
    inter_ymax = tf.minimum(t_ymax, p_ymax)
    
    inter_area = tf.maximum(0.0, inter_xmax - inter_xmin) * tf.maximum(0.0, inter_ymax - inter_ymin)
    true_area = (t_xmax - t_xmin) * (t_ymax - t_ymin)
    pred_area = (p_xmax - p_xmin) * (p_ymax - p_ymin)
    union_area = true_area + pred_area - inter_area
    iou = inter_area / tf.maximum(union_area, 1e-6)
    
    t_cx = (t_xmin + t_xmax) / 2.0
    t_cy = (t_ymin + t_ymax) / 2.0
    p_cx = (p_xmin + p_xmax) / 2.0
    p_cy = (p_ymin + p_ymax) / 2.0
    center_dist_sq = tf.square(t_cx - p_cx) + tf.square(t_cy - p_cy)
    
    enclosing_xmin = tf.minimum(t_xmin, p_xmin)
    enclosing_ymin = tf.minimum(t_ymin, p_ymin)
    enclosing_xmax = tf.maximum(t_xmax, p_xmax)
    enclosing_ymax = tf.maximum(t_ymax, p_ymax)
    enc_diag_sq = tf.square(enclosing_xmax - enclosing_xmin) + tf.square(enclosing_ymax - enclosing_ymin)
    
    penalty = center_dist_sq / tf.maximum(enc_diag_sq, 1e-6)
    return 1.0 - iou + penalty

model = build_detector()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={'bounding_box': diou_loss, 'class_label': 'categorical_crossentropy'},
    metrics={'class_label': 'accuracy'}
)

# =====================================================================
# 4. CONSTRUCT DATA PIPELINE
# =====================================================================
def get_filepaths(images_dir, labels_dir):
    img_files = sorted([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    img_paths = [os.path.join(images_dir, f) for f in img_files]
    lbl_paths = [os.path.join(labels_dir, os.path.splitext(f)[0] + '.txt') for f in img_files]
    return img_paths, lbl_paths

train_img_paths, train_lbl_paths = get_filepaths(train_images_dir, train_labels_dir)
val_img_paths, val_lbl_paths = get_filepaths(val_images_dir, val_labels_dir)

def parse_element_safe(img_path, lbl_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img = img / 255.0
    
    label_content = tf.io.read_file(lbl_path)
    lines = tf.strings.split(label_content, '\n')
    num_lines = tf.shape(lines)[0]
    
    def process_valid_label():
        first_line = tf.strings.split(lines[0], ' ')
        return tf.cond(
            tf.greater_equal(tf.shape(first_line)[0], 5),
            lambda: (
                tf.strings.to_number(first_line[0], out_type=tf.int32),
                tf.strings.to_number(first_line[1], out_type=tf.float32),
                tf.strings.to_number(first_line[2], out_type=tf.float32),
                tf.strings.to_number(first_line[3], out_type=tf.float32),
                tf.strings.to_number(first_line[4], out_type=tf.float32)
            ),
            lambda: (0, 0.0, 0.0, 0.0, 0.0)
        )
        
    class_id, x_center, y_center, w, h = tf.cond(
        tf.greater(num_lines, 0),
        process_valid_label,
        lambda: (0, 0.0, 0.0, 0.0, 0.0)
    )
    
    xmin = x_center - (w / 2.0)
    ymin = y_center - (h / 2.0)
    xmax = x_center + (w / 2.0)
    ymax = y_center + (h / 2.0)
    
    box_target = tf.stack([xmin, ymin, xmax, ymax])
    class_target = tf.cond(tf.equal(class_id, 0), lambda: tf.constant([1.0, 0.0]), lambda: tf.constant([0.0, 1.0]))
    return img, {'bounding_box': box_target, 'class_label': class_target}

train_dataset = tf.data.Dataset.from_tensor_slices((train_img_paths, train_lbl_paths))
train_dataset = train_dataset.map(parse_element_safe, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=100).batch(32).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((val_img_paths, val_lbl_paths))
val_dataset = val_dataset.map(parse_element_safe, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

print(f"Pipeline initialized cleanly. Ready for training...")

# =====================================================================
# 5. EXECUTE TRAINING
# =====================================================================
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    verbose=1
)

model.save('/kaggle/working/tf_obstacle_detector.h5')
print("\n[SUCCESS] Upgraded model saved cleanly to /kaggle/working/tf_obstacle_detector.h5!")

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

print("Rebuilding architecture into a Spatial Grid-Based Multi-Box Custom Detector...")

# =====================================================================
# 1. ARCHITECTURE DEFINITION (8x8 Spatial Grid)
# =====================================================================
def build_grid_detector():
    base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    
    inputs = layers.Input(shape=(224, 224, 3))
    features = base_model(inputs, training=False)
    
    # Custom Convolutional Head to process local spatial grid regions (7x7 or 8x8)
    x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(features)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    # Head 1: Grid-wise Box Regression (Grid size: 7x7, 4 coordinates per cell)
    # Output shape: (None, 7, 7, 4)
    box_output = layers.Conv2D(4, (1, 1), activation='sigmoid', name='grid_bounding_box')(x)
    
    # Head 2: Grid-wise Multi-Class Classification (Grid size: 7x7, 2 classes per cell)
    # Output shape: (None, 7, 7, 2)
    class_output = layers.Conv2D(2, (1, 1), activation='softmax', name='grid_class_label')(x)
    
    return models.Model(inputs=inputs, outputs=[box_output, class_output])

# =====================================================================
# 2. CUSTOM DATA COUPLING TO THE GRID
# =====================================================================
base_path = '/kaggle/input/datasets/abtinzandi/obstacle-detection-dataset/ROD-Dataset/dataset'
train_images_dir = os.path.join(base_path, 'train/images')
train_labels_dir = os.path.join(base_path, 'train/labels')

img_files = sorted([f for f in os.listdir(train_images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
train_img_paths = [os.path.join(train_images_dir, f) for f in img_files]
train_lbl_paths = [os.path.join(train_labels_dir, os.path.splitext(f)[0] + '.txt') for f in img_files]

def parse_element_to_grid(img_path, lbl_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img = img / 255.0
    
    label_content = tf.io.read_file(lbl_path)
    lines = tf.strings.split(label_content, '\n')
    
    # Default empty grids (7x7 map matching MobileNet output dimensions)
    box_grid = tf.zeros((7, 7, 4), dtype=tf.float32)
    class_grid = tf.tensor_scatter_nd_update(
        tf.zeros((7, 7, 2), dtype=tf.float32), 
        indices=[[i, j] for i in range(7) for j in range(7)], 
        updates=[[1.0, 0.0] for _ in range(49)] # Default background/Class 0
    )
    
    # Dynamic parsing to assign target object coordinates to its exact grid cell center
    first_line = tf.strings.split(lines[0], ' ')
    if tf.greater_equal(tf.shape(first_line)[0], 5):
        class_id = tf.strings.to_number(first_line[0], out_type=tf.int32)
        x_center = tf.strings.to_number(first_line[1], out_type=tf.float32)
        y_center = tf.strings.to_number(first_line[2], out_type=tf.float32)
        w = tf.strings.to_number(first_line[3], out_type=tf.float32)
        h = tf.strings.to_number(first_line[4], out_type=tf.float32)
        
        # Calculate exactly which grid cell (0 to 6) contains the center of the obstacle
        grid_x = tf.clip_by_value(tf.cast(x_center * 7.0, tf.int32), 0, 6)
        grid_y = tf.clip_by_value(tf.cast(y_center * 7.0, tf.int32), 0, 6)
        
        # Update coordinates and classification target dynamically inside that specific block
        box_grid = tf.tensor_scatter_nd_update(box_grid, [[grid_y, grid_x]], [[x_center, y_center, w, h]])
        class_target = tf.cond(tf.equal(class_id, 0), lambda: tf.constant([1.0, 0.0]), lambda: tf.constant([0.0, 1.0]))
        class_grid = tf.tensor_scatter_nd_update(class_grid, [[grid_y, grid_x]], [class_target])
        
    return img, {'grid_bounding_box': box_grid, 'grid_class_label': class_grid}

train_dataset = tf.data.Dataset.from_tensor_slices((train_img_paths, train_lbl_paths))
train_dataset = train_dataset.map(parse_element_to_grid, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(100).batch(32).prefetch(tf.data.AUTOTUNE)

model = build_grid_detector()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={'grid_bounding_box': 'mean_squared_error', 'grid_class_label': 'categorical_crossentropy'}
)

print("Grid System Initialized! Execute model.fit() now to train the custom localized heads.")

In [ ]:
# =====================================================================
# 1. TRAIN THE GRID MODEL
# =====================================================================
print("Training the grid detector over 15 epochs...")
history = model.fit(
    train_dataset,
    epochs=15,
    verbose=1
)

model.save('/kaggle/working/tf_grid_obstacle_detector.h5')
print("\n[SUCCESS] Grid model trained successfully!")

# =====================================================================
# 2. RUN LOCALIZED INFERENCE ON YOUR VIDEO
# =====================================================================
video_path = '/kaggle/input/datasets/abuzarjamil/test-video/WhatsApp Video 2026-06-22 at 11.05.49 PM.mp4'
cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS)) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30

output_path = '/kaggle/working/grid_output_detected.mp4'
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

print("\nProcessing frames using grid parsing. Filtering out the human class...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (224, 224))
    img_batch = np.expand_dims(img_resized.astype(np.float32) / 255.0, axis=0)
    
    # Predict the grid maps
    box_grid_preds, class_grid_preds = model.predict(img_batch, verbose=0)
    
    # Extract the 7x7 maps from the batch
    box_map = box_grid_preds[0]    # Shape: (7, 7, 4)
    class_map = class_grid_preds[0]  # Shape: (7, 7, 2)
    
    # Scan through all 49 cells of the spatial grid
    for row in range(7):
        for col in range(7):
            classes = class_map[row, col]
            class_idx = np.argmax(classes)
            confidence = classes[class_idx]
            
            # CRITICAL FILTER: Index 1 represents the Cone obstacle in our pipeline setup.
            # If the cell registers 'Cone' with high confidence, draw it. Ignore class_idx == 0 (Person).
            if class_idx == 1 and confidence > 0.4:
                x_center, y_center, w, h = box_map[row, col]
                
                # Turn normalized grid relative coordinates back into video pixels
                xmin = x_center - (w / 2.0)
                ymin = y_center - (h / 2.0)
                xmax = x_center + (w / 2.0)
                ymax = y_center + (h / 2.0)
                
                start_point = (int(xmin * frame_width), int(ymin * frame_height))
                end_point = (int(xmax * frame_width), int(ymax * frame_height))
                
                # Draw a tight box only on the localized cone grid location
                cv2.rectangle(frame, start_point, end_point, (0, 165, 255), 3)
                cv2.putText(frame, f"Obstacle: Cone {confidence:.2f}", 
                            (start_point[0], max(start_point[1] - 10, 20)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)
                
    out.write(frame)

cap.release()
out.release()
print("\n[SUCCESS] Localized tracking complete! Download your video at: /kaggle/working/grid_output_detected.mp4")